# 13 — Teacher pretraining and exact recovery

AHR v2 learns broader Go knowledge from the pinned `quintic/go9x9` corpus converted by notebook 12. Scout energies learn legal teacher move distributions. Worker energies learn **teacher estimates**, with observed game winners kept as a separate target. Score and ownership losses apply only where those labels are valid. This is supervised learning from KataGo-generated data, not a claim of bounded verification or playing strength.

**Inputs:** tagged definitions in 00–02 and 12; immutable, compatible train/development caches. The official test split is untouched. **Outputs:** `checkpoints/<run_id>/` (latest two and best full resumable states), `reports/ahr_v2/<run_id>/` metrics/status, and validation reports. **Runtime:** Run All runs small CPU/GPU recovery checks; long pretraining is launched separately in detached tmux by notebook 15. **Storage:** the persistent boot disk; the user has deferred off-VM backup.

Every sampler permutation and cursor, D4 augmentation RNG, model, optimizer, scheduler, scaler, CPU/GPU RNG, source/data/config identity and fixed development sample is saved. Corrupt or mismatched runs refuse to restart. The same player API serves the larger configurable residual network.

**Validated:** fresh Run All passed on real converted rows. CPU FP32 and GPU 1 BF16 recovery matched bit for bit across independent fresh kernels, including D4 augmentation and an epoch transition. This validates training recovery, not playing strength.

In [ ]:
from pathlib import Path
import os, nbformat
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT: raise RuntimeError('Open in the AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(cell.source,globals())
for name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','12_teacher_dataset_and_targets.ipynb'):
    go2_import_exports(PROJECT/GO2_ROOT/name,globals())
print('Imported definitions only; upstream generation and training cells were not executed.')

## Configuration and identity

The production default uses GPU 1, a wider 96-channel six-block network and mixed BF16 computation. Soft teacher value is mapped to loss/win probabilities because the dataset supplies a signed scalar, not a full draw distribution. That approximation is recorded explicitly. Whole-game winners are distinct observed labels; no tactical or cutoff result is converted into a full-game winner.

Observed game outcomes are audited separately; their training weight is zero by default because the recorded game result can disagree with the teacher estimate. The default sampler randomizes the order of 16,384-row blocks and every row within each block, covering the complete bank each epoch while keeping disk reads efficient.

In [ ]:
GO2_TEACHER_TRAINING_NOTEBOOK='13_teacher_pretraining_and_recovery.ipynb'
GO2_TEACHER_DATA_NOTEBOOK='12_teacher_dataset_and_targets.ipynb'

def go2_teacher_training_config(run_id='ahr-v2-teacher-20261008',cache_id='go9x9-teacher-v1',device='cuda:1',time_budget_hours=8):
    return {'schema':'ahr-go-v2-teacher-training-v1','run_id':run_id,'cache_id':cache_id,'device':device,
        'time_budget_hours':float(time_budget_hours),'seed':260810,'board_size':9,'network':{'width':96,'blocks':6},
        'batch_size':512,'shuffle_block_rows':16384,'learning_rate':0.0005,'weight_decay':0.0001,'gradient_clip':2.0,
        'max_optimizer_steps':100000,'max_epochs':12,'warmup_steps':200,'minimum_lr_fraction':0.1,
        'precision':'bfloat16' if str(device).startswith('cuda') else 'float32','cpu_threads':4,
        'checkpoint_seconds':120,'checkpoint_optimizer_steps':100,'evaluate_every_steps':500,
        'development_rows':2048,'train_rows_limit':None,'is_smoke':False,
        'loss_weights':{'policy':1.0,'teacher_value':1.0,'game_outcome':0.0,'score':0.5,'ownership':0.25},
        'value_target_mapping':'signed_teacher_scalar_to_soft_loss_win_no_draw_claim',
        'training_source':'quintic_go9x9_katago_teacher_and_recorded_game_outcomes'}

def go2_validate_teacher_training_config(config):
    base=go2_teacher_training_config()
    if set(config)-set(base):raise ValueError('Unknown teacher training settings: '+str(sorted(set(config)-set(base))))
    config={**base,**copy.deepcopy(config)}
    for key in ('schema','value_target_mapping','training_source'):
        if config[key]!=base[key]:raise ValueError('Unsupported teacher target schema: '+key)
    for key in ('run_id','cache_id'):
        if not config[key] or Path(config[key]).name!=config[key]:raise ValueError('Local directory name required: '+key)
    if config['device'] not in ('cpu','cuda:0','cuda:1'):raise ValueError('Use CPU or an explicit CUDA device; production defaults to GPU 1')
    for key in ('batch_size','shuffle_block_rows','max_optimizer_steps','max_epochs','cpu_threads','checkpoint_optimizer_steps','evaluate_every_steps','development_rows'):
        if type(config[key]) is not int or config[key]<1:raise ValueError('Positive integer required: '+key)
    if config['board_size']!=9:raise ValueError('This converted corpus is native 9x9')
    if config['train_rows_limit'] is not None and (type(config['train_rows_limit']) is not int or config['train_rows_limit']<1):raise ValueError('Invalid training subset')
    for key in ('learning_rate','gradient_clip','checkpoint_seconds','time_budget_hours'):
        if not math.isfinite(config[key]) or config[key]<=0:raise ValueError('Positive finite setting required: '+key)
    if config['checkpoint_seconds']>900 or config['checkpoint_optimizer_steps']>250:raise ValueError('Checkpoint cadence exceeds workspace agreement')
    if config['warmup_steps']<0 or not 0<config['minimum_lr_fraction']<=1:raise ValueError('Invalid learning-rate schedule')
    if config['precision'] not in ('float32','bfloat16') or (config['device']=='cpu' and config['precision']!='float32'):raise ValueError('Unsupported precision/device combination')
    if set(config['loss_weights'])!=set(base['loss_weights']) or any(not math.isfinite(v) or v<0 for v in config['loss_weights'].values()):raise ValueError('Invalid explicit target weights')
    Go2NetConfig(board_size=9,**config['network'])
    return config

def go2_teacher_training_sources(project):
    names=('00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb',GO2_TEACHER_DATA_NOTEBOOK,GO2_TEACHER_TRAINING_NOTEBOOK)
    return {GO2_ROOT+'/'+name:go2_code_sha(Path(project)/GO2_ROOT/name) for name in names}

def go2_teacher_split_count(dataset,split):
    item=dataset[split]
    if isinstance(item,dict):
        for key in ('count','row_count','rows','n_rows'):
            if key in item and isinstance(item[key],(int,np.integer)):return int(item[key])
    if isinstance(item,(int,np.integer)):return int(item)
    raise RuntimeError('Dataset split must expose an integer row count: '+split)

def go2_teacher_data_identity(dataset):
    manifest=dataset['manifest']
    return {'manifest_sha256':go2_digest(manifest),'cache_id':manifest.get('cache_id',str(dataset.get('cache_path',''))),
        'dataset':manifest.get('dataset','quintic/go9x9'),'target_provenance':'teacher_estimates_and_distinct_observed_outcomes'}

def go2_teacher_training_identity(project,config,dataset):
    return {'schema':GO2_SCHEMA,'sources':go2_teacher_training_sources(project),'config':copy.deepcopy(config),
        'data':go2_teacher_data_identity(dataset),'torch':torch.__version__,'numpy':np.__version__,'rules':GO2_RULE_PROFILE}

def go2_teacher_determinism(config):
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
    torch.set_num_threads(config['cpu_threads']);torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    if config['device'].startswith('cuda:') and (not torch.cuda.is_available() or int(config['device'].split(':')[1])>=torch.cuda.device_count()):raise RuntimeError('Requested GPU unavailable; refusing device fallback')

def go2_teacher_cuda_devices(config):
    return (int(config['device'].split(':')[1]),) if config['device'].startswith('cuda:') else ()

def go2_teacher_permutation(rows,generator,block_rows):
    # Shuffle the full bank while keeping mmap reads within a bounded hot set.
    blocks=(rows+block_rows-1)//block_rows
    order=torch.randperm(blocks,generator=generator)
    pieces=[]
    for block in order.tolist():
        start=block*block_rows;count=min(block_rows,rows-start)
        pieces.append(torch.randperm(count,generator=generator)+start)
    return torch.cat(pieces)

def go2_teacher_lr_factor(step,config):
    warmup=config['warmup_steps']
    if warmup and step<warmup:return (step+1)/warmup
    fraction=min(max((step-warmup)/max(1,config['max_optimizer_steps']-warmup),0.0),1.0)
    minimum=config['minimum_lr_fraction']
    return minimum+(1-minimum)*0.5*(1+math.cos(math.pi*fraction))

## Targets, masks and symmetry

Each batch carries legal policy probabilities, teacher value and optional labels with separate validity masks. Illegal policy mass is rejected after notebook 12 conversion. D4 augmentation transforms every board plane, board action, legal mask and ownership point together; pass, value, score, player and komi keep their meanings.

In [ ]:
def go2_teacher_validate_batch(batch):
    required={'features','legal','policy','value','score','score_valid','outcome','outcome_valid','ownership','ownership_valid'}
    if required-set(batch):raise ValueError('Incomplete typed batch: '+str(sorted(required-set(batch))))
    n=int(batch['features'].shape[0]);expected={'features':(n,21,9,9),'legal':(n,82),'policy':(n,82),'value':(n,),
        'score':(n,),'score_valid':(n,),'outcome':(n,3),'outcome_valid':(n,),'ownership_valid':(n,)}
    for key,shape in expected.items():
        if tuple(batch[key].shape)!=shape:raise ValueError('Target shape mismatch: '+key)
    if tuple(batch['ownership'].shape) not in ((n,9,9),(n,81)):raise ValueError('Ownership must contain 81 points')
    for key in ('features','policy','value','score','outcome','ownership'):
        if not torch.isfinite(batch[key]).all():raise ValueError('Non-finite batch field: '+key)
    legal=batch['legal'].bool();policy=batch['policy'].float()
    if not legal.any(1).all() or (policy<0).any() or (policy.masked_select(~legal).abs()>1e-6).any():raise ValueError('Invalid legal policy probabilities')
    if not torch.allclose(policy.sum(1),torch.ones(n,device=policy.device),atol=0.002,rtol=0):raise ValueError('Policy targets must sum to one')
    if (batch['value'].abs()>1.00001).any() or (batch['score'][batch['score_valid'].bool()].abs()>1.00001).any():raise ValueError('Normalized value/score outside bounds')
    valid=batch['outcome_valid'].bool();outcome=batch['outcome'].float()
    if valid.any() and ((outcome[valid]<0).any() or not torch.allclose(outcome[valid].sum(1),torch.ones(int(valid.sum()),device=outcome.device),atol=0.002,rtol=0)):raise ValueError('Observed outcome probabilities invalid')
    return batch

def go2_teacher_d4(batch,symmetries):
    result=dict(batch);device=batch['features'].device
    symmetries=torch.as_tensor(symmetries,dtype=torch.long,device=device)
    if tuple(symmetries.shape)!=(batch['features'].shape[0],) or ((symmetries<0)|(symmetries>7)).any():raise ValueError('One D4 symmetry per position required')
    inverse=[]
    for symmetry in range(8):
        inv=[0]*81
        for old in range(81):inv[go2_transform_action(old,9,symmetry)]=old
        inverse.append(inv)
    permutation=torch.tensor(inverse,dtype=torch.long,device=device)[symmetries]
    n=len(symmetries)
    result['features']=batch['features'].reshape(n,21,81).gather(2,permutation[:,None,:].expand(-1,21,-1)).reshape(n,21,9,9)
    for key in ('legal','policy'):
        result[key]=torch.cat((batch[key][:,:81].gather(1,permutation),batch[key][:,81:]),dim=1)
    result['ownership']=batch['ownership'].reshape(n,81).gather(1,permutation).reshape(n,9,9)
    return result

def go2_teacher_masked_mean(values,mask):
    mask=mask.bool()
    return values[mask].mean() if bool(mask.any()) else values.sum()*0.0

def go2_teacher_joint_loss(model,batch,config):
    output=model(batch['features'])
    logits=(-output['scout_energy'].float()/model.config.scout_temperature).masked_fill(~batch['legal'].bool(),-1e9)
    policy_log=F.log_softmax(logits,dim=1)
    value_log=F.log_softmax(-output['worker_energy'].float()/model.config.worker_temperature,dim=1)
    value=batch['value'].float().clamp(-1,1)
    soft_value=torch.stack(((1-value)/2,torch.zeros_like(value),(1+value)/2),dim=1)
    components={'policy':-(batch['policy'].float()*policy_log).sum(1).mean(),
        'teacher_value':-(soft_value*value_log).sum(1).mean(),
        'game_outcome':go2_teacher_masked_mean(-(batch['outcome'].float()*value_log).sum(1),batch['outcome_valid']),
        'score':go2_teacher_masked_mean((output['score'].float()-batch['score'].float()).square(),batch['score_valid']),
        'ownership':go2_teacher_masked_mean((output['ownership'].float()-batch['ownership'].float().reshape(-1,81)).square().mean(1),batch['ownership_valid'])}
    loss=sum(config['loss_weights'][key]*item for key,item in components.items())
    if not torch.isfinite(loss):raise RuntimeError('Non-finite supervised objective')
    return {'loss':loss,'components':components,'output':output,'policy_log':policy_log,'value_log':value_log}

def go2_teacher_evaluate(model,dataset,indices,config):
    was_training=model.training;model.eval();totals={};rows=0;outcome_rows=0;score_rows=0
    try:
        with torch.inference_mode():
            for offset in range(0,len(indices),config['batch_size']):
                ids=indices[offset:offset+config['batch_size']]
                batch=go2_teacher_sample_batch(dataset,ids,device=config['device'],split='dev')
                go2_teacher_validate_batch(batch)
                result=go2_teacher_joint_loss(model,batch,config);n=len(ids);rows+=n
                policy=batch['policy'].float();pred=result['policy_log'].exp();value_pred=result['value_log'].exp()
                entropy=-(policy*policy.clamp_min(1e-12).log()).sum(1)
                additions={'joint_loss':result['loss']*n,'policy_cross_entropy':result['components']['policy']*n,
                    'policy_kl':((policy*(policy.clamp_min(1e-12).log()-result['policy_log'])).sum(1)).sum(),
                    'policy_top1_agreement':(pred.argmax(1)==policy.argmax(1)).float().sum(),
                    'teacher_value_mse':((value_pred[:,2]-value_pred[:,0]-batch['value']).square()).sum(),
                    'teacher_win_probability_brier':((value_pred[:,2]-(1+batch['value'])/2).square()).sum()}
                for key,item in additions.items():totals[key]=totals.get(key,0.0)+float(item.cpu())
                valid=batch['outcome_valid'].bool();outcome_rows+=int(valid.sum())
                if valid.any():
                    totals['observed_outcome_brier']=totals.get('observed_outcome_brier',0.0)+float((value_pred[valid]-batch['outcome'][valid]).square().sum().cpu())
                    totals['observed_outcome_accuracy']=totals.get('observed_outcome_accuracy',0.0)+float((value_pred[valid].argmax(1)==batch['outcome'][valid].argmax(1)).float().sum().cpu())
                valid=batch['score_valid'].bool();score_rows+=int(valid.sum())
                if valid.any():totals['teacher_score_mae_points']=totals.get('teacher_score_mae_points',0.0)+float((result['output']['score'][valid].float()-batch['score'][valid]).abs().sum().cpu())*88.5
    finally:model.train(was_training)
    if not rows:raise RuntimeError('Empty development set')
    report={key:value/(outcome_rows if key.startswith('observed_') else score_rows if key=='teacher_score_mae_points' else rows) for key,value in totals.items()}
    report.update(rows=rows,observed_outcome_rows=outcome_rows,teacher_score_rows=score_rows,
        selection_score=report['joint_loss'],selection_basis='fixed_official_development_joint_loss_not_playing_strength')
    return report

## Training and interruption recovery

A checkpoint is committed at each completed epoch, each development evaluation, every 100 optimizer updates or two minutes, and on a graceful pause. `STOP` under the run checkpoint directory requests a pause between optimizer updates. A partially failed update is discarded; the previous valid checkpoint remains authoritative.

In [ ]:
def go2_teacher_progress(state,entry,status):
    metrics=state['metrics'];sampler=state['sampler']
    return {'schema':'ahr-go-v2-teacher-progress-v1','run_id':state['config']['run_id'],'status':status,
        'updated_at_utc':go2_now(),'optimizer_step':state['global_step'],'epoch':sampler['epoch'],'epoch_cursor':sampler['cursor'],
        'training_rows':sampler['train_rows'],'rows_seen':sampler['rows_seen'],'active_seconds':state['active_seconds'],
        'budget_hours':state['config']['time_budget_hours'],'last_loss':metrics['last_loss'],'last_development':metrics['last_development'],
        'best_development_joint_loss':metrics['best_development_joint_loss'],'checkpoint':entry,
        'data_identity':state['identity']['data'],'learning_source':state['config']['training_source'],
        'target_claim':'Teacher predictions are estimates; observed winners remain distinct; no test rows used'}

def go2_run_teacher_training(project=None,config=None,max_steps=None,on_checkpoint=None):
    project=go2_project(project);config=go2_validate_teacher_training_config(config or go2_teacher_training_config())
    go2_teacher_determinism(config)
    dataset=go2_teacher_dataset(project,cache_id=config['cache_id'],verify=True)
    train_rows=go2_teacher_split_count(dataset,'train');dev_rows=go2_teacher_split_count(dataset,'dev')
    if config['train_rows_limit'] is not None:train_rows=min(train_rows,config['train_rows_limit'])
    if train_rows<1 or dev_rows<1:raise RuntimeError('Both train and independent development data are required')
    identity=go2_teacher_training_identity(project,config,dataset);directory=project/'checkpoints'/config['run_id']
    report_dir=project/'reports/ahr_v2'/config['run_id'];report_dir.mkdir(parents=True,exist_ok=True)
    with go2_checkpoint_lock(directory):
        saved,entry=go2_load_checkpoint(directory,identity)
        go2_seed(config['seed'])
        model=Go2Network(Go2NetConfig(board_size=9,**config['network'])).to(config['device'])
        optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=config['weight_decay'])
        scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:go2_teacher_lr_factor(step,config))
        scaler=torch.amp.GradScaler('cuda',enabled=False)
        sampler=torch.Generator(device='cpu').manual_seed(config['seed']+101)
        dev_generator=torch.Generator(device='cpu').manual_seed(config['seed']+303)
        development_indices=torch.randperm(dev_rows,generator=dev_generator)[:min(config['development_rows'],dev_rows)]
        state={'schema':GO2_SCHEMA,'identity':identity,'config':config,'global_step':0,'stage':'teacher_pretraining',
            'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':{},'active_seconds':0.0,
            'sampler':{'epoch':0,'cursor':0,'rows_seen':0,'train_rows':train_rows,'permutation':go2_teacher_permutation(train_rows,sampler,config['shuffle_block_rows'])},
            'development_indices':development_indices,'data_identity':identity['data'],
            'metrics':{'optimizer_step':0,'last_loss':None,'loss_ema':None,'best_development_joint_loss':None,
                'last_development':None,'last_evaluation_step':-1,'recent_losses':[],'development_history':[],
                'target_rows_seen':{'policy':0,'teacher_value':0,'score':0,'ownership':0,'observed_outcome':0}}}
        if saved is not None:
            state=saved;model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            go2_restore_rng(state['rng'],{'sampler':sampler});go2_invalidate_model_identity(model)
            if state['sampler']['train_rows']!=train_rows or len(state['sampler']['permutation'])!=train_rows:raise RuntimeError('Saved sampler/data extent differs')
        go2_atomic_json(directory/'config.json',config)
        last_saved=-1 if entry is None else state['global_step'];last_checkpoint=time.monotonic();steps_saved=state['global_step']
        processed=0;status='RUNNING';pending_best=False
        def write_progress():
            progress=go2_teacher_progress(state,entry,status)
            go2_atomic_json(directory/'progress.json',progress);go2_atomic_json(report_dir/'status.json',progress)
        def save_snapshot(force=False):
            nonlocal entry,last_saved,last_checkpoint,steps_saved,pending_best
            if state['global_step']==last_saved:return
            if not force and time.monotonic()-last_checkpoint<config['checkpoint_seconds'] and state['global_step']-steps_saved<config['checkpoint_optimizer_steps']:return
            state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),
                scheduler=copy.deepcopy(scheduler.state_dict()),scaler=copy.deepcopy(scaler.state_dict()),
                rng=go2_capture_rng(cuda_devices=go2_teacher_cuda_devices(config),generators={'sampler':sampler}))
            entry=go2_save_checkpoint(directory,state,best=pending_best,
                resume_cell='13/15: go2_run_teacher_training(PROJECT,json.loads((PROJECT/"checkpoints"/run_id/"config.json").read_text()))')
            pending_best=False;last_saved=state['global_step'];last_checkpoint=time.monotonic();steps_saved=state['global_step']
            if on_checkpoint is not None:
                before=go2_capture_rng(cuda_devices=go2_teacher_cuda_devices(config),generators={'sampler':sampler})
                try:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
                finally:go2_restore_rng(before,{'sampler':sampler})
        def evaluate():
            nonlocal pending_best
            before=go2_capture_rng(cuda_devices=go2_teacher_cuda_devices(config),generators={'sampler':sampler})
            started=time.monotonic()
            try:audit=go2_teacher_evaluate(model,dataset,state['development_indices'],config)
            finally:go2_restore_rng(before,{'sampler':sampler})
            audit['optimizer_step']=state['global_step'];audit['duration_seconds']=time.monotonic()-started
            metrics=state['metrics'];metrics['last_development']=audit;metrics['last_evaluation_step']=state['global_step']
            metrics['development_history']=(metrics['development_history']+[audit])[-256:]
            old=metrics['best_development_joint_loss']
            if old is None or audit['selection_score']<old:metrics['best_development_joint_loss']=audit['selection_score'];pending_best=True
            with (report_dir/'development.jsonl').open('a') as stream:stream.write(json.dumps(audit)+'\n');stream.flush()
        if saved is None:evaluate()
        save_snapshot(True);write_progress()
        try:
            while True:
                if (directory/'STOP').exists():status='STOPPED';break
                if state['active_seconds']>=config['time_budget_hours']*3600:status='TIME_BUDGET_COMPLETE';break
                if state['global_step']>=config['max_optimizer_steps']:status='STEP_LIMIT_COMPLETE';break
                if state['sampler']['epoch']+int(state['sampler']['cursor']==train_rows)>=config['max_epochs']:status='EPOCH_LIMIT_COMPLETE';break
                if max_steps is not None and processed>=int(max_steps):status='PAUSED';break
                started=time.monotonic();sampler_state=state['sampler']
                if sampler_state['cursor']==train_rows:
                    sampler_state['epoch']+=1;sampler_state['cursor']=0;sampler_state['permutation']=go2_teacher_permutation(train_rows,sampler,config['shuffle_block_rows'])
                start=sampler_state['cursor'];stop=min(start+config['batch_size'],train_rows)
                ids=sampler_state['permutation'][start:stop]
                batch=go2_teacher_sample_batch(dataset,ids,device=config['device'],split='train')
                go2_teacher_validate_batch(batch)
                symmetries=torch.randint(0,8,(len(ids),),generator=sampler)
                batch=go2_teacher_d4(batch,symmetries)
                model.train();optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda' if config['device'].startswith('cuda') else 'cpu',dtype=torch.bfloat16,enabled=config['precision']=='bfloat16'):
                    result=go2_teacher_joint_loss(model,batch,config)
                result['loss'].backward();torch.nn.utils.clip_grad_norm_(model.parameters(),config['gradient_clip'],error_if_nonfinite=True)
                optimizer.step();scheduler.step();go2_invalidate_model_identity(model)
                sampler_state['cursor']=stop;sampler_state['rows_seen']+=len(ids);state['global_step']+=1;processed+=1
                metrics=state['metrics'];metrics['optimizer_step']=state['global_step'];metrics['last_loss']=float(result['loss'].detach().cpu())
                metrics['loss_ema']=metrics['last_loss'] if metrics['loss_ema'] is None else 0.95*metrics['loss_ema']+0.05*metrics['last_loss']
                record={'optimizer_step':state['global_step'],'rows':len(ids),'loss':metrics['last_loss'],
                    'components':{key:float(value.detach().cpu()) for key,value in result['components'].items()},'learning_rate':optimizer.param_groups[0]['lr'],
                    'target_rows':{'policy':len(ids),'teacher_value':len(ids),'score':int(batch['score_valid'].sum()),
                        'ownership':int(batch['ownership_valid'].sum()),'observed_outcome':int(batch['outcome_valid'].sum())}}
                for key,count in record['target_rows'].items():metrics['target_rows_seen'][key]+=count
                metrics['recent_losses']=(metrics['recent_losses']+[record])[-256:]
                state['active_seconds']+=time.monotonic()-started
                with (report_dir/'metrics.jsonl').open('a') as stream:stream.write(json.dumps(record)+'\n');stream.flush()
                epoch_boundary=sampler_state['cursor']==train_rows
                needs_evaluation=state['global_step']-metrics['last_evaluation_step']>=config['evaluate_every_steps'] or epoch_boundary
                if needs_evaluation:evaluate()
                save_snapshot(epoch_boundary or needs_evaluation)
                if state['global_step']%20==0 or epoch_boundary or needs_evaluation:write_progress()
        except BaseException as error:
            try:
                valid,valid_entry=go2_load_checkpoint(directory,identity)
                failed=go2_teacher_progress(valid,valid_entry,'ERROR') if valid is not None else {'run_id':config['run_id'],'status':'ERROR'}
            except Exception as checkpoint_error:failed={'run_id':config['run_id'],'status':'ERROR','checkpoint_validation_error':repr(checkpoint_error)}
            failed.update(error=repr(error),discarded_partial_optimizer_update=True,updated_at_utc=go2_now())
            go2_atomic_json(directory/'progress.json',failed);go2_atomic_json(report_dir/'status.json',failed)
            raise
        if status in ('TIME_BUDGET_COMPLETE','STEP_LIMIT_COMPLETE','EPOCH_LIMIT_COMPLETE') and state['global_step']!=state['metrics']['last_evaluation_step'] and state['global_step']!=last_saved:evaluate()
        save_snapshot(True);write_progress()
        return state

def go2_load_teacher_player(project=None,run_id=None,which='latest',device='cpu'):
    project=go2_project(project)
    if run_id is None:
        pointer=project/'reports/ahr_v2/teacher_active_run.json'
        if not pointer.exists():raise FileNotFoundError('No active teacher run; provide run_id')
        run_id=json.loads(pointer.read_text())['run_id']
    directory=project/'checkpoints'/str(run_id)
    for attempt in range(4):
        before=(directory/'manifest.json').read_bytes()
        try:state,entry=go2_load_checkpoint(directory,which=which)
        except (RuntimeError,FileNotFoundError):
            after=(directory/'manifest.json').read_bytes()
            if before==after or attempt==3:raise
            continue
        break
    if state is None:raise FileNotFoundError('No teacher checkpoint')
    if state['identity']['sources']!=go2_teacher_training_sources(project):raise RuntimeError('Teacher checkpoint source identity changed')
    if state['config']['schema']!='ahr-go-v2-teacher-training-v1':raise RuntimeError('Wrong training schema')
    with torch.random.fork_rng(devices=[]):model=Go2Network(Go2NetConfig(board_size=9,**state['config']['network']))
    model.load_state_dict(state['model']);model.to(device);model.eval();model.requires_grad_(False)
    metadata={'run_id':str(run_id),'global_step':state['global_step'],'optimizer_step':state['global_step'],
        'model_identity':go2_model_identity(model),'checkpoint_file':entry['file'],'checkpoint_sha256':entry['sha256'],
        'config':state['config'],'source_identity':state['identity'],'sources':state['identity']['sources'],
        'data_identity':state['identity']['data'],'training_source':state['config']['training_source'],
        'is_smoke':state['config']['is_smoke'],'best_selection_basis':'fixed_development_joint_loss_not_playing_strength',
        'completed_games':None,'unknown_games':None,'development':state['metrics']['last_development'],
        'target_rows_seen':state['metrics']['target_rows_seen'],'ownership_trained':state['metrics']['target_rows_seen']['ownership']>0}
    return model,metadata

## Exact recovery checks

The fixtures use real converted dataset rows, including D4 augmentation and an epoch transition. Continuous and interrupted training must match model, optimizer, schedule, scaler, permutation/cursor, development sample, learning metrics and every used RNG bit for bit. Wall-clock measurements are excluded from scientific equality.

In [ ]:
def go2_teacher_scientific_state(state):
    result={key:copy.deepcopy(state[key]) for key in ('model','optimizer','scheduler','scaler','rng','sampler','global_step','development_indices','metrics')}
    for audit in result['metrics']['development_history']:audit.pop('duration_seconds',None)
    if result['metrics']['last_development'] is not None:result['metrics']['last_development'].pop('duration_seconds',None)
    return result

def go2_teacher_fresh_kernel_worker(project,config,steps,label):
    from nbclient import NotebookClient
    directory=Path(project)/'reports/ahr_v2'/config['run_id'];directory.mkdir(parents=True,exist_ok=True)
    path=directory/('recovery-'+label+'.ipynb')
    bootstrap="from pathlib import Path\nimport os,nbformat,json\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nPROJECT=Path("+repr(str(project))+" )\nfor cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:\n    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())\nfor name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','12_teacher_dataset_and_targets.ipynb','13_teacher_pretraining_and_recovery.ipynb'):\n    go2_import_exports(PROJECT/GO2_ROOT/name,globals())"
    run="CONFIG=json.loads("+repr(json.dumps(config))+" )\nSTATE=go2_run_teacher_training(PROJECT,CONFIG,max_steps="+str(int(steps))+" )\nprint({'optimizer_step':STATE['global_step'],'epoch':STATE['sampler']['epoch'],'cursor':STATE['sampler']['cursor']})"
    worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('Disposable exact-recovery fixture; independent fresh kernel.'),nbformat.v4.new_code_cell(bootstrap),nbformat.v4.new_code_cell(run)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(worker,path)
    NotebookClient(worker,timeout=240,kernel_name='ahr-go',resources={'metadata':{'path':str(project)}}).execute()
    nbformat.write(worker,path)
    state,_=go2_load_checkpoint(Path(project)/'checkpoints'/config['run_id'])
    if state is None:raise AssertionError('Fresh kernel worker produced no checkpoint')
    return state

def go2_teacher_resume_fixture(project,cache_id,device):
    token=uuid.uuid4().hex[:10];base=go2_teacher_training_config('teacher-fixture-'+token+'-continuous',cache_id,device,0.05)
    base.update(network={'width':8,'blocks':1},batch_size=16,max_optimizer_steps=6,warmup_steps=0,train_rows_limit=80,
        development_rows=32,evaluate_every_steps=2,checkpoint_optimizer_steps=2,is_smoke=True,precision='bfloat16' if device.startswith('cuda') else 'float32')
    continuous=go2_teacher_fresh_kernel_worker(project,base,6,'continuous')
    resumed_config={**copy.deepcopy(base),'run_id':'teacher-fixture-'+token+'-resumed'}
    stopped=go2_teacher_fresh_kernel_worker(project,resumed_config,3,'interrupted')
    if stopped['sampler']['cursor']!=48 or stopped['global_step']!=3:raise AssertionError('Fixture did not stop with a pending sampler permutation')
    resumed=go2_teacher_fresh_kernel_worker(project,resumed_config,3,'resumed')
    if not go2_nested_equal(go2_teacher_scientific_state(continuous),go2_teacher_scientific_state(resumed)):raise AssertionError('Exact teacher optimizer recovery differs: '+device)
    run_dir=Path(project)/'checkpoints'/resumed_config['run_id'];manifest,states=go2_manifest_states(run_dir)
    if len(states)<2 or manifest['best'] not in states:raise AssertionError('Retention lost resumable checkpoints')
    model,metadata=go2_load_teacher_player(project,resumed_config['run_id'],device='cpu')
    if go2_tensor_digest(model.state_dict())!=go2_tensor_digest(resumed['model']):raise AssertionError('Player does not load exact checkpoint weights')
    wrong=copy.deepcopy(resumed['identity']);wrong['data']['manifest_sha256']='0'*64
    try:go2_load_checkpoint(run_dir,wrong)
    except RuntimeError:pass
    else:raise AssertionError('Data identity mismatch accepted')
    return {'device':device,'precision':base['precision'],'steps':resumed['global_step'],'rows_seen':resumed['sampler']['rows_seen'],
        'epoch':resumed['sampler']['epoch'],'cursor':resumed['sampler']['cursor'],'retained_checkpoints':len(states),
        'model_sha256':go2_tensor_digest(resumed['model']),'exact_scientific_state':True,'fresh_kernel_per_invocation':True,'data_mismatch_refused':True,
        'continuous_run_id':base['run_id'],'resumed_run_id':resumed_config['run_id']}

def go2_validate_teacher_training(project=None,cache_id='go9x9-teacher-smoke-v1'):
    project=go2_project(project);checks=[]
    state=go2_initial_state(9,7.5)
    for action in (40,30,41,31,50,49):state=go2_step(state,action)
    encoded=torch.tensor(go2_encode_state(state))[None]
    dummy={'features':encoded,'legal':torch.tensor(go2_legal_mask(state))[None],
        'policy':torch.tensor(go2_legal_mask(state),dtype=torch.float32)[None],
        'value':torch.tensor([0.25]),'score':torch.tensor([0.1]),'score_valid':torch.tensor([True]),
        'outcome':torch.tensor([[0.,0.,1.]]),'outcome_valid':torch.tensor([True]),
        'ownership':torch.arange(81,dtype=torch.float32).reshape(1,9,9),'ownership_valid':torch.tensor([True])}
    dummy['policy']/=dummy['policy'].sum(1,keepdim=True)
    for symmetry in range(8):
        transformed=go2_teacher_d4(dummy,[symmetry]);exact=go2_transform_state(state,symmetry)
        if not torch.equal(transformed['features'][0],torch.tensor(go2_encode_state(exact))):raise AssertionError('D4 features differ from legal-state encoding')
        if not torch.equal(transformed['legal'][0],torch.tensor(go2_legal_mask(exact))):raise AssertionError('D4 legal mask differs')
        expected=torch.tensor(go2_transform_vector(dummy['policy'][0].tolist(),9,symmetry))
        if not torch.equal(transformed['policy'][0],expected):raise AssertionError('D4 policy differs')
        expected=torch.tensor(go2_transform_vector(range(81),9,symmetry),dtype=torch.float32).reshape(9,9)
        if not torch.equal(transformed['ownership'][0],expected):raise AssertionError('D4 ownership differs')
        if transformed['value'].item()!=0.25 or transformed['policy'][0,-1]!=dummy['policy'][0,-1]:raise AssertionError('D4 altered scalar/pass')
    checks.append({'name':'D4 matches exact legal-state features/actions/ownership all eight symmetries','passed':True})
    dataset=go2_teacher_dataset(project,cache_id=cache_id,verify=True)
    sample=go2_teacher_sample_batch(dataset,torch.arange(min(32,go2_teacher_split_count(dataset,'train'))),split='train',device='cpu')
    go2_teacher_validate_batch(sample);checks.append({'name':'real compatible teacher batch targets/masks','passed':True,'rows':len(sample['value'])})
    fixtures=[go2_teacher_resume_fixture(project,cache_id,'cpu')]
    if torch.cuda.is_available() and torch.cuda.device_count()>=2:fixtures.append(go2_teacher_resume_fixture(project,cache_id,'cuda:1'))
    else:raise RuntimeError('GPU1 exact recovery required before a GPU long run')
    # Strict corruption refusal is tested on a disposable copy, never the valid checkpoint.
    import tempfile,shutil
    source=project/'checkpoints'/fixtures[0]['resumed_run_id']
    with tempfile.TemporaryDirectory(prefix='teacher-corruption-',dir=project/'checkpoints') as temp:
        bad=Path(temp);shutil.copytree(source,bad,dirs_exist_ok=True)
        manifest=json.loads((bad/'manifest.json').read_text());payload=bad/manifest['latest']
        raw=bytearray(payload.read_bytes());raw[len(raw)//2]^=1;payload.write_bytes(raw)
        try:go2_load_checkpoint(bad)
        except RuntimeError:checks.append({'name':'corrupt retained payload refuses fallback/restart','passed':True})
        else:raise AssertionError('Corrupt checkpoint accepted')
    report={'schema':'ahr-go-v2-teacher-training-validation-v1','updated_at_utc':go2_now(),'checks':checks,'fixtures':fixtures,
        'cache_id':cache_id,'source_identity':go2_teacher_training_sources(project),'all_passed':True}
    go2_atomic_json(project/'reports/ahr_v2/teacher_training_validation.json',report)
    return report

In [ ]:
VALIDATION=go2_validate_teacher_training(PROJECT)
import pandas as pd
pd.DataFrame(VALIDATION['fixtures'])[['device','steps','rows_seen','epoch','cursor','retained_checkpoints','exact_scientific_state']]

## Progress and exact resume

The chart below shows only disposable recovery fixtures. Production progress belongs to the run reports and notebook 15; teacher agreement alone does not establish Go playing strength. Independent balanced GNU Go matches remain the playing benchmark.

**Exact resume:** import tagged definitions, load the saved `checkpoints/<run_id>/config.json`, then call `go2_run_teacher_training(PROJECT, saved_config)`. If a graceful STOP was intentional, remove `checkpoints/<run_id>/STOP` before that explicit resume. Never edit a saved run configuration to increase its budget or change its dataset; create a new versioned run instead.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(9,3.5),'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
fixture_id=VALIDATION['fixtures'][-1]['continuous_run_id']
fixture_state,_=go2_load_checkpoint(PROJECT/'checkpoints'/fixture_id)
losses=fixture_state['metrics']['recent_losses'];audits=fixture_state['metrics']['development_history']
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
axes[0].plot([r['optimizer_step'] for r in losses],[r['loss'] for r in losses],marker='o',label='Training batch')
axes[0].plot([r['optimizer_step'] for r in audits],[r['joint_loss'] for r in audits],marker='s',linestyle='--',label='Fixed development')
axes[0].set(title='Recovery fixture: supervised objective',xlabel='Optimizer update',ylabel='Weighted loss');axes[0].legend()
axes[1].plot([r['optimizer_step'] for r in audits],[r['policy_top1_agreement'] for r in audits],marker='s',label='Teacher move agreement')
axes[1].plot([r['optimizer_step'] for r in audits],[r['teacher_win_probability_brier'] for r in audits],marker='o',linestyle='--',label='Teacher win-probability Brier')
axes[1].set(title='Recovery fixture: held-out predictions',xlabel='Optimizer update',ylabel='Fraction / mean squared error');axes[1].legend()
for ax in axes:ax.grid(alpha=0.2)
fig.tight_layout();plt.show()
print('Validation passed on real converted rows with exact CPU and GPU recovery. These six-update fixtures do not measure playing strength.')